# Download Data

This example notebook shows you how to download water data time-series from Google Earthengine for 
1. Annual JRC surface water data
2. Dynamic World - monthly aggregated data

### Imports

In [ ]:
from water_timeseries.downloader import EarthEngineDownloader
import geopandas as gpd

#### Loading vector dataset
* we are loading the provided lake polygon dataset
* the specific unique lake id is "id_geohash"
* explore the dataset

In [2]:
vector_path = '../tests/data/lake_polygons.parquet'
gdf = gpd.read_parquet(vector_path)

In [3]:
gdf.head()

,label_id,id,Area_start_ha,Area_end_ha,NetChange_ha,NetChange_perc,GrossIncrease_ha,GrossIncrease_perc,GrossDecrease_ha,GrossDecrease_perc,...,id_geohash,river,source_file,staging_duplicated,area_vector_ha,area_ratio,num_holes,holes_per_ha,__index_level_0__,geometry
0,18644,18644,2.7009,2.8008,0.0999,3.698766,0.2043,7.564145,0.1044,3.727506,...,b7g6g1ny1mf7,0.0,lake_change_32603,False,2.684576,1.043293,0,0.0,61180,"POLYGON ((-164.03911 66.58432, -164.03912 66.5..."
1,18659,18659,1.0242,1.3743,0.3501,34.182775,0.5184,50.615114,0.1683,12.246235,...,b7g4yc12k4yj,0.0,lake_change_32603,False,1.341851,1.024182,0,0.0,61187,"POLYGON ((-164.23263 66.5835, -164.23264 66.58..."
2,18655,18655,4.4370,4.7583,0.3213,7.241379,0.7614,17.160244,0.4401,9.249102,...,b7g6c8gye56e,0.0,lake_change_32603,False,4.651507,1.022959,0,0.0,61188,"POLYGON ((-164.10951 66.58339, -164.10952 66.5..."
3,18667,18667,4.5747,4.4649,-0.1098,-2.400158,0.3312,7.239819,0.4410,9.877041,...,b7g4v0vsnux0,0.0,lake_change_32603,False,5.009950,0.913123,0,0.0,61192,"POLYGON ((-164.30774 66.58277, -164.30775 66.5..."
4,18670,18670,1.1349,1.0899,-0.0450,-3.965107,0.0234,2.061856,0.0684,6.275805,...,b7g6u8dbyu5m,0.0,lake_change_32603,False,1.252487,0.906117,0,0.0,61195,"POLYGON ((-163.97973 66.58044, -163.97974 66.5..."


In [4]:
gdf.explore(width=1200, height=600)

#### Setup GEE account
* set project name to EE_PROJECT environment variable (preferred)

In [ ]:
import os
os.environ['EE_PROJECT'] = 'YOUR_GEE_PROJECT'

#### Setup logger (optional)
* here I am using loguru, you can also choose others

In [6]:
import loguru
logger = loguru.logger
logger.add("logfile_dl.log", level=0)

1

#### Download JRC dataset
* requires polygon vector dataset + GEE account
* returns xarray dataset

#### Simple case JRC
* Download JRC annual timeseries for
  * default years (2000-2021)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: area_land, area_nodata, area_water_permanent, area_water_seasonal      

In [7]:
dl = EarthEngineDownloader(ee_project="pdg-wg-workflow", ee_auth=True, logger=logger)
ds_jrc = dl.download_jrc_annual(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    n_parallel=1,
)

2026-04-09 21:09:21.607 | INFO     | water_timeseries.downloader:_log_info:115 - Initializing EarthEngineDownloader with project: pdg-wg-workflow
2026-04-09 21:09:24.084 | INFO     | water_timeseries.downloader:_log_info:115 - EarthEngineDownloader initialized successfully. Output directory: downloads
2026-04-09 21:09:24.086 | INFO     | water_timeseries.utils.io:load_vector_dataset:39 - Loading vector dataset from ../tests/data/lake_polygons.parquet
2026-04-09 21:09:24.099 | INFO     | water_timeseries.downloader:_log_info:115 - Initial dataset has 118 features
2026-04-09 21:09:24.100 | INFO     | water_timeseries.downloader:_log_info:115 - No ID filtering applied
2026-04-09 21:09:24.100 | INFO     | water_timeseries.downloader:_log_info:115 - No spatial bbox filtering applied (using default global bounds)
2026-04-09 21:09:24.101 | INFO     | water_timeseries.downloader:_log_info:115 - Processing 118 features
2026-04-09 21:09:24.102 | INFO     | water_timeseries.downloader:_log_info:1

In [8]:
ds_jrc

<xarray.Dataset> Size: 84kB
Dimensions:               (id_geohash: 118, date: 22)
Coordinates:
  * id_geohash            (id_geohash) object 944B 'b7g4hv14r3u8' ... 'b7g6y0...
  * date                  (date) datetime64[ns] 176B 2000-01-01 ... 2021-01-01
Data variables:
    area_land             (id_geohash, date) float64 21kB 0.2614 ... 0.05553
    area_nodata           (id_geohash, date) float64 21kB 0.0 0.04303 ... 0.0
    area_water_permanent  (id_geohash, date) float64 21kB 1.43 1.313 ... 0.1797
    area_water_seasonal   (id_geohash, date) float64 21kB 0.2148 0.0 ... 0.1256

#### Simple case Dynamic World
* Download Dynamic World monthly timeseries for
  * default years (2016-2025)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: bare, built, crops, flooded_vegetation, grass, shrub_and_scrub, snow_and_ice, trees, water

In [9]:
ds_dw = dl.download_dw_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    years=range(2017, 2026),
    months=[6, 7, 8, 9],
    n_parallel=1,
)

2026-04-09 21:09:38.862 | INFO     | water_timeseries.utils.io:load_vector_dataset:39 - Loading vector dataset from ../tests/data/lake_polygons.parquet
2026-04-09 21:09:38.874 | INFO     | water_timeseries.downloader:_log_info:115 - Initial dataset has 118 features
2026-04-09 21:09:38.875 | INFO     | water_timeseries.downloader:_log_info:115 - No ID filtering applied
2026-04-09 21:09:38.875 | INFO     | water_timeseries.downloader:_log_info:115 - No spatial bbox filtering applied (using default global bounds)
2026-04-09 21:09:38.876 | INFO     | water_timeseries.downloader:_log_info:115 - Processing 118 features
2026-04-09 21:09:38.877 | INFO     | water_timeseries.downloader:_log_info:115 - Processing 118 features x 36 dates = 4248 total requests
2026-04-09 21:09:38.877 | INFO     | water_timeseries.downloader:_log_info:115 - Processing dates: ['2017-06-01', '2017-07-01', '2017-08-01', '2017-09-01', '2018-06-01', '2018-07-01', '2018-08-01', '2018-09-01', '2019-06-01', '2019-07-01', '

In [10]:
ds_dw

<xarray.Dataset> Size: 307kB
Dimensions:             (id_geohash: 118, date: 36)
Coordinates:
  * id_geohash          (id_geohash) object 944B 'b7g4hv14r3u8' ... 'b7g6y07s...
  * date                (date) object 288B '2017-06-01T00:00:00' ... '2025-09...
Data variables:
    bare                (id_geohash, date) float64 34kB 0.0 0.0 0.0 ... 0.0 0.0
    built               (id_geohash, date) float64 34kB 0.0 0.0 0.0 ... 0.0 0.0
    crops               (id_geohash, date) float64 34kB 0.0 0.0 0.0 ... 0.0 0.0
    flooded_vegetation  (id_geohash, date) float64 34kB 0.0 0.0 0.0 ... 0.0 0.0
    grass               (id_geohash, date) float64 34kB 0.0 0.0 ... 0.1335 0.0
    shrub_and_scrub     (id_geohash, date) float64 34kB 0.0 0.0 ... 0.2102
    snow_and_ice        (id_geohash, date) float64 34kB 2.251 0.0 ... 0.0 0.0
    trees               (id_geohash, date) float64 34kB 0.0 0.0 ... 0.02459 0.0
    water               (id_geohash, date) float64 34kB 0.0 0.0 ... 0.2805 0.24